# ESMT Rankings Intelligence — minimal cron test

This notebook tests one narrow end-to-end path:

**GitHub Actions cron → live collection → CSV/state files → Streamlit**

To make failures easy to diagnose, only one technically simple source is enabled: the official Corporate Knights WordPress API. There is no paid AI, semantic search, ranking history, or complex scraping in this checkpoint.

Collection rules:

- first successful run: request the previous **90 days**;
- later runs: request records since the last successful run, with a **48-hour safety overlap**;
- deduplicate by canonical URL;
- retain stored records for **365 days**;
- make collection failure stop the workflow instead of silently reporting “no news”.


In [ ]:
from pathlib import Path
from urllib.parse import urlencode, urlparse, urlunparse, parse_qsl
from urllib.request import Request, urlopen
from datetime import datetime, timezone
import hashlib
import html
import json
import re

import pandas as pd
from IPython.display import display


DATA_DIR = Path("data")
NEWS_PATH = DATA_DIR / "news.csv"
STATUS_PATH = DATA_DIR / "collector_status.csv"
STATE_PATH = DATA_DIR / "run_state.json"

INITIAL_BACKFILL_DAYS = 90
RETENTION_DAYS = 365
SAFETY_OVERLAP_HOURS = 48
REQUEST_TIMEOUT_SECONDS = 30
NOW_UTC = pd.Timestamp.now(tz="UTC")

SOURCE = {
    "source_id": "corporate_knights",
    "publisher": "Corporate Knights",
    "source_group": "ranking",
    "api_url": "https://corporateknights.com/wp-json/wp/v2/posts",
    "official_domain": "corporateknights.com",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run time: {NOW_UTC.isoformat()}")
print(f"Enabled source: {SOURCE['publisher']}")


## 1. Decide whether this is the first backfill or an incremental run

The state is stored in `data/run_state.json`. The scheduled workflow commits this file back to GitHub after every successful run, so the next run can continue from the recorded timestamp.


In [ ]:
def load_state(path):
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return {}


state = load_state(STATE_PATH)
last_successful_value = state.get("last_successful_run_utc")
last_successful_run = (
    pd.to_datetime(last_successful_value, utc=True, errors="coerce")
    if last_successful_value
    else pd.NaT
)

first_run = pd.isna(last_successful_run)
if first_run:
    collection_since = NOW_UTC - pd.Timedelta(days=INITIAL_BACKFILL_DAYS)
    collection_mode = "initial_90_day_backfill"
else:
    collection_since = last_successful_run - pd.Timedelta(hours=SAFETY_OVERLAP_HOURS)
    collection_mode = "incremental_with_48h_overlap"

print(f"Mode: {collection_mode}")
print(f"Request publications after: {collection_since.isoformat()}")


## 2. Collect live records

The source is queried server-side with an `after` timestamp. Pagination is supported, but no browser automation or HTML-page crawling is needed.


In [ ]:
TRACKING_KEYS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "fbclid", "gclid", "mc_cid", "mc_eid",
}


def canonicalize_url(value):
    parsed = urlparse(str(value).strip())
    query = [
        (key, item)
        for key, item in parse_qsl(parsed.query, keep_blank_values=True)
        if key.lower() not in TRACKING_KEYS
    ]
    path = re.sub(r"/{2,}", "/", parsed.path or "/")
    return urlunparse((parsed.scheme.lower(), parsed.netloc.lower(), path, "", urlencode(query), ""))


def clean_html(value):
    text = re.sub(r"<[^>]+>", " ", value or "")
    return re.sub(r"\s+", " ", html.unescape(text)).strip()


def collect_wordpress_posts(source, since):
    base_params = {
        "per_page": 100,
        "after": since.isoformat(),
        "orderby": "date",
        "order": "desc",
        "_fields": "id,date_gmt,link,title,excerpt",
    }

    all_items = []
    page = 1
    total_pages = 1

    while page <= total_pages:
        params = dict(base_params, page=page)
        request_url = f"{source['api_url']}?{urlencode(params)}"
        request = Request(
            request_url,
            headers={"User-Agent": "ESMT-ranking-intelligence-cron-test/1.0", "Accept": "application/json"},
        )
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            page_items = json.loads(response.read().decode("utf-8"))
            total_pages = int(response.headers.get("X-WP-TotalPages", "1"))
        all_items.extend(page_items)
        page += 1

    records = []
    for item in all_items:
        url = canonicalize_url(item.get("link", ""))
        host = (urlparse(url).hostname or "").lower()
        if host != source["official_domain"] and not host.endswith("." + source["official_domain"]):
            continue

        title = clean_html((item.get("title") or {}).get("rendered", ""))
        if not title or not url:
            continue

        records.append({
            "article_id": hashlib.sha1(url.encode("utf-8")).hexdigest()[:16],
            "source_id": source["source_id"],
            "publisher": source["publisher"],
            "source_group": source["source_group"],
            "title": title,
            "excerpt": clean_html((item.get("excerpt") or {}).get("rendered", "")),
            "url": url,
            "published_at": pd.to_datetime(item.get("date_gmt"), utc=True, errors="coerce"),
            "collected_at": NOW_UTC,
        })

    return records


collection_error = ""
try:
    live_records = collect_wordpress_posts(SOURCE, collection_since)
    collection_status = "ok"
except Exception as exc:
    live_records = []
    collection_status = "error"
    collection_error = f"{type(exc).__name__}: {exc}"

print(f"Collection status: {collection_status}")
print(f"Fetched records: {len(live_records)}")
if collection_error:
    print(collection_error)


## 3. Apply transparent Important rules

These rules are intentionally simple and inspectable. They can be expanded after the online execution is proven reliable.


In [ ]:
IMPORTANT_RULES = {
    "ranking_release": re.compile(r"\b(?:ranking|rankings|ranked|league table|top\s+\d+)\b", re.I),
    "methodology_or_weights": re.compile(r"\b(?:methodology|indicator|metric|weight|weighting|formula)\w*\b", re.I),
    "eligibility_or_deadline": re.compile(r"\b(?:eligibility|eligible|participation|submission|deadline|calendar)\b", re.I),
    "accreditation": re.compile(r"\b(?:accreditation|accredited|reaccredited|AACSB|AMBA|EQUIS|ZEvA)\b", re.I),
}
ESMT_PATTERN = re.compile(
    r"\b(?:ESMT(?:\s+Berlin)?|European\s+School\s+of\s+Management\s+(?:and|&)\s+Technology)\b",
    re.I,
)


def important_reasons(text):
    reasons = [name for name, pattern in IMPORTANT_RULES.items() if pattern.search(text)]
    if ESMT_PATTERN.search(text):
        reasons.insert(0, "esmt_mention")
    return reasons


new_frame = pd.DataFrame(live_records)
if not new_frame.empty:
    searchable = new_frame[["title", "excerpt"]].fillna("").agg(" ".join, axis=1)
    reason_lists = searchable.map(important_reasons)
    new_frame["important_reasons"] = reason_lists.map(lambda values: " | ".join(values))
    new_frame["is_important"] = reason_lists.map(bool)
else:
    new_frame = pd.DataFrame(columns=[
        "article_id", "source_id", "publisher", "source_group", "title", "excerpt",
        "url", "published_at", "collected_at", "important_reasons", "is_important",
    ])

display(new_frame[["published_at", "title", "is_important", "important_reasons"]].head(10))


## 4. Merge, deduplicate and persist

The existing CSV remains the source of history. A daily run rechecks a small overlap, replaces duplicate URLs with the freshest copy, and then removes records older than 365 days.


In [ ]:
EXPECTED_COLUMNS = [
    "article_id", "source_id", "publisher", "source_group", "title", "excerpt",
    "url", "published_at", "collected_at", "important_reasons", "is_important",
]

if NEWS_PATH.exists():
    existing_frame = pd.read_csv(NEWS_PATH)
    for date_column in ["published_at", "collected_at"]:
        existing_frame[date_column] = pd.to_datetime(existing_frame[date_column], utc=True, errors="coerce")
else:
    existing_frame = pd.DataFrame(columns=EXPECTED_COLUMNS)

existing_urls = set(existing_frame.get("url", pd.Series(dtype=str)).dropna())
new_item_count = int((~new_frame.get("url", pd.Series(dtype=str)).isin(existing_urls)).sum())

if existing_frame.empty:
    combined = new_frame.copy()
elif new_frame.empty:
    combined = existing_frame.copy()
else:
    combined = pd.concat([existing_frame, new_frame], ignore_index=True)
combined = combined.drop_duplicates("url", keep="last")
retention_cutoff = NOW_UTC - pd.Timedelta(days=RETENTION_DAYS)
combined = combined[
    combined["published_at"].isna() | (combined["published_at"] >= retention_cutoff)
].copy()
combined = combined.sort_values("published_at", ascending=False, na_position="last")
combined = combined[EXPECTED_COLUMNS]
combined.to_csv(NEWS_PATH, index=False)

status_frame = pd.DataFrame([{
    "run_at_utc": NOW_UTC.isoformat(),
    "source_id": SOURCE["source_id"],
    "publisher": SOURCE["publisher"],
    "status": collection_status,
    "collection_mode": collection_mode,
    "collection_since_utc": collection_since.isoformat(),
    "fetched_items": len(new_frame),
    "new_items": new_item_count,
    "stored_items": len(combined),
    "error": collection_error,
}])
status_frame.to_csv(STATUS_PATH, index=False)

if collection_status == "ok":
    new_state = {
        "last_successful_run_utc": NOW_UTC.isoformat(),
        "last_collection_mode": collection_mode,
        "stored_items": len(combined),
    }
    STATE_PATH.write_text(json.dumps(new_state, indent=2), encoding="utf-8")

display(status_frame)
print(f"Stored articles: {len(combined)}")
print(f"New articles this run: {new_item_count}")
print(f"Important articles stored: {int(combined['is_important'].astype(str).str.lower().eq('true').sum())}")

if collection_status != "ok":
    raise RuntimeError(f"Live collection failed: {collection_error}")


## What success looks like online

After the workflow runs successfully, GitHub should contain a new bot commit with:

- `data/news.csv`;
- `data/collector_status.csv`;
- `data/run_state.json`.

The first run records `initial_90_day_backfill`. The next run records `incremental_with_48h_overlap`. The Streamlit app reads these files directly; no API is required for this infrastructure test.
